In [1]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f3003877 (23F3003877-t12026) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from IPython.display import Audio, display

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchaudio
import torch.optim as optim 
import torchaudio.functional as F
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"device: {device}")

from glob import glob
import random
from tqdm import tqdm
import soundfile as sf

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import os
import sys
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks.early_stopping import EarlyStopping

from sklearn.metrics import f1_score , accuracy_score, classification_report, confusion_matrix
import math

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("github_access")

!rm -rf /kaggle/working/dl-genai-project-26-t1

!git clone https://{secret_value_0}@github.com/Aryanch9797/dl-genai-project-26-t1.git

repo_path = "/kaggle/working/dl-genai-project-26-t1"
if repo_path not in sys.path:
    sys.path.append(repo_path)

from src.Trainers.custom_trainer import training_model
from src.dataset.test_dataset_for_AST import test_mashed_dataset
from src.dataset.train_val_dataset_for_AST import MelSpectrogramDataset
from src.models.AST import AST_model
from src.inference import prediction

Cloning into 'dl-genai-project-26-t1'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 328 (delta 114), reused 91 (delta 38), pack-reused 130 (from 1)
Receiving objects: 100% (328/328), 64.45 MiB | 33.37 MiB/s, done.
Resolving deltas: 100% (182/182), done.


In [2]:
val_path ="/kaggle/input/notebooks/aryanchauhan97971234/data-generator-ipynb/train_data/"
train_paths = [
    "/kaggle/input/notebooks/aryanchauhan97971234/data-generator-2-ipynb/train_data/",
    "/kaggle/input/notebooks/aryanchauhan97971234/data-generator-3-ipynb/train_data/",
    "/kaggle/input/notebooks/aryanchauhan97971234/data-generator-4-ipynb/train_data/",
    "/kaggle/input/notebooks/aryanchauhan97971234/data-generator-5-ipynb/train_data/"
]

In [3]:
train_dataset = MelSpectrogramDataset(False, train_paths, val_path)
val_dataset = MelSpectrogramDataset(True, train_paths, val_path)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4,pin_memory=True,persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4,pin_memory=True,persistent_workers=True)

In [4]:
model = AST_model(lr=1e-3)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([10])          
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([10, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [6]:
checkpoint_callback = ModelCheckpoint(
    monitor='val/f1_macro',    
    dirpath='saved_models/', 
    filename='AST-best-{epoch:02d}-{val/f1_macro:.2f}',
    save_top_k=1,            
    mode='max'            
)

early_stop_callback = EarlyStopping(
    monitor='val/f1_macro', 
    min_delta=0.00,         
    patience=5,             
    verbose=True,
    mode='max'              
)

wandb_logger = WandbLogger(project="Audio_Mashup", name="AST_full_v3")
wandb_logger.experiment.define_metric("epoch")
wandb_logger.experiment.define_metric("train/*", step_metric="epoch")
wandb_logger.experiment.define_metric("val/*", step_metric="epoch")


trainer = pl.Trainer(
    max_epochs=40, 
    accelerator="gpu", 
    devices=1,  
    logger=wandb_logger,
    callbacks=[checkpoint_callback, early_stop_callback] 
)
trainer.fit(model, train_loader, val_loader)
best_ckpt_path = checkpoint_callback.best_model_path
checkpoint = torch.load(best_ckpt_path)
best_state_dict = checkpoint['state_dict']

In [ ]:

import kagglehub

MODEL_UPLOAD_DIR = "/kaggle/working/AST" 
os.makedirs(MODEL_UPLOAD_DIR, exist_ok=True)

MODEL_SAVE_PATH = os.path.join(MODEL_UPLOAD_DIR, "AST.pth")
torch.save(best_state_dict, MODEL_SAVE_PATH)     # model_trained
print(f"Model saved to {MODEL_SAVE_PATH}")

KAGGLE_USERNAME = 'aryanchauhan97971234' 
MODEL = 'AST'
FRAMEWORK = 'pytorch'
VARIATION = 'full_tuned_AST'

handle = f'{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}'

print(f"Uploading model from {MODEL_UPLOAD_DIR} to {handle}...")

kagglehub.model_upload(
    handle,                     
    MODEL_UPLOAD_DIR,           
    license_name="Apache 2.0", 
    version_notes="4th run with transfer, lr scheduler"
)
print("Upload complete!")